# 추세와 계절성 분해

> 파이썬 15강 · 시계열과 예측

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [추세와 계절성 분해](https://mioon1402.github.io/timeseriesdata/python/p15-decompose.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 시계열을 이루는 세 조각

**15-1. 데이터 준비**

In [ ]:
import pandas as pd

df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])
s = df.set_index("date").asfreq("D")["sales"].interpolate()

print("기간:", s.index[0].date(), "~", s.index[-1].date())
print("일수:", len(s), "  결측:", s.isna().sum())

## 2. 주간 계절성 분해하기

**15-2. seasonal_decompose**

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

분해 = seasonal_decompose(s, model="additive", period=7)

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
최근 = slice("2025-06", "2025-09")     # 4개월만 확대해서 보자

axes[0].plot(s.loc[최근] / 10000, lw=0.8); axes[0].set_ylabel("원본")
axes[1].plot(분해.trend.loc[최근] / 10000, lw=2, color="#dc2626"); axes[1].set_ylabel("추세")
axes[2].plot(분해.seasonal.loc[최근] / 10000, lw=1, color="#0d9488"); axes[2].set_ylabel("계절성")
axes[3].plot(분해.resid.loc[최근] / 10000, lw=0.6, color="#94a3b8"); axes[3].set_ylabel("나머지")
axes[3].axhline(0, color="black", lw=0.6)

fig.suptitle("매출 = 추세 + 계절성 + 나머지 (단위: 만원)", fontweight="bold")
plt.tight_layout()
plt.show()

**15-3. 각 조각의 크기 비교**

In [ ]:
추세폭 = 분해.trend.dropna()
print(f"추세      {추세폭.iloc[0]:>10,.0f}원 → {추세폭.iloc[-1]:>10,.0f}원 "
      f"(2년간 +{추세폭.iloc[-1]/추세폭.iloc[0]*100-100:.0f}%)")
print(f"계절성 진폭 {분해.seasonal.max() - 분해.seasonal.min():>10,.0f}원")
print(f"나머지 SD   {분해.resid.dropna().std():>10,.0f}원")
print()
print(f"원본 SD     {s.std():>10,.0f}원")

## 3. 계절성분 읽는 법

**15-4. 요일별 계절성분**

In [ ]:
요일이름 = ["월", "화", "수", "목", "금", "토", "일"]
계절 = pd.Series(분해.seasonal.values, index=분해.seasonal.index.dayofweek)
요일별 = 계절.groupby(level=0).mean()
요일별.index = 요일이름

print("요일별 계절성분 (평균 대비 만원)")
for 요일, 값 in (요일별 / 10000).round(1).items():
    막대 = "█" * int(abs(값) / 2)
    부호 = "+" if 값 > 0 else " "
    print(f"  {요일}  {부호}{값:>6.1f}  {막대}")

**15-5. 나머지로 이상한 날 찾기**

In [ ]:
나머지 = 분해.resid.dropna()
한계 = 3 * 나머지.std()

이상한날 = 나머지[abs(나머지) > 한계].sort_values()
print(f"나머지가 3σ({한계:,.0f}원)를 넘은 날: {len(이상한날)}일\n")

for 날짜, 값 in 이상한날.items():
    요일 = 요일이름[날짜.dayofweek]
    print(f"  {날짜.date()} ({요일})  {값/10000:+8.1f}만원")

## 4. 가법이냐 승법이냐

**15-6. 승법 분해로 비교**

In [ ]:
import numpy as np

양수 = s.replace(0, np.nan).interpolate()     # 승법은 0을 허용하지 않는다
승법 = seasonal_decompose(양수, model="multiplicative", period=7)

지수 = pd.Series(승법.seasonal.values, index=승법.seasonal.index.dayofweek)
요일지수 = 지수.groupby(level=0).mean()
요일지수.index = 요일이름

print("요일별 계절지수 (1.0 = 평균)")
for 요일, 값 in 요일지수.round(3).items():
    print(f"  {요일}  {값:.3f}   ({(값-1)*100:+5.1f}%)")

## 5. 연간 계절성 분해

**15-7. 월 단위 연간 분해**

In [ ]:
월평균 = s.resample("ME").mean()
연간 = seasonal_decompose(월평균, model="additive", period=12)

월별계절 = pd.Series(연간.seasonal.values, index=월평균.index.month)
월별계절 = 월별계절.groupby(level=0).mean() / 10000

print("월별 계절성분 (평균 대비 만원)")
for 월, 값 in 월별계절.round(1).items():
    막대 = "█" * int(abs(값))
    print(f"  {월:>2}월  {값:>+6.1f}  {막대}")

## 6. 계절조정 — 진짜 성장 보기

**15-8. 계절조정 시계열**

In [ ]:
조정 = s - 분해.seasonal          # 가법이므로 빼면 된다

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(s.index, s / 10000, lw=0.4, color="#cbd5e1", label="원본")
ax.plot(조정.index, 조정 / 10000, lw=0.7, color="#2563eb", label="계절조정 (요일 효과 제거)")
ax.plot(분해.trend.index, 분해.trend / 10000, lw=2.5, color="#dc2626", label="추세")

ax.set_ylabel("매출(만원)")
ax.set_title("요일 효과를 걷어내면 성장 흐름이 선명해진다",
             fontweight="bold", loc="left")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

**15-9. 조정 전후 비교**

In [ ]:
print(f"원본       표준편차 {s.std():>10,.0f}원")
print(f"계절조정   표준편차 {조정.std():>10,.0f}원")
print(f"→ 요일 효과가 전체 변동의 {(1 - 조정.std()/s.std())*100:.0f}% 를 차지")

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 방문객(visitors)을 주기 7로 분해하고, 요일별 계절성분을 구해보세요.


# 문제 2. 승법 모형의 나머지(resid)와 가법 모형의 나머지를 비교해보세요.
#        어느 쪽이 더 작은가요?  (승법의 resid 는 1 근처가 기준입니다)


# 문제 3. 계절조정한 매출로 '전월 대비' 변화율을 구해보세요.
#        원본으로 구한 것과 어떻게 다른가요?

**모범 답안**

In [ ]:
v = df.set_index("date").asfreq("D")["visitors"].interpolate()

# 문제 1
분해v = seasonal_decompose(v, model="additive", period=7)
계절v = pd.Series(분해v.seasonal.values, index=분해v.seasonal.index.dayofweek)
결과 = 계절v.groupby(level=0).mean().round(1)
결과.index = 요일이름
print("요일별 계절성분 (명)")
print(결과.to_string())

# 문제 2
가법잔차 = 분해.resid.dropna()
승법잔차 = 승법.resid.dropna()
print(f"\n가법 나머지 SD (원)      {가법잔차.std():,.0f}")
print(f"승법 나머지 SD (비율)    {승법잔차.std():.4f}  "
      f"→ 평균 대비 약 {승법잔차.std()*100:.1f}%")
print(f"가법을 비율로 환산하면    {가법잔차.std()/s.mean()*100:.1f}%")

# 문제 3
조정월 = (s - 분해.seasonal).resample("ME").mean()
원본월 = s.resample("ME").mean()
비교 = pd.DataFrame({
    "원본 전월대비(%)":   원본월.pct_change() * 100,
    "조정 전월대비(%)":   조정월.pct_change() * 100,
}).round(2)
print()
print(비교.tail(6).to_string())

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)